In [1]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f'device used: PC GPU - {torch.cuda.get_device_name(0)}')
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"device used: MAC GPU - MPS")
else:
    device = torch.device("cpu")
    print(f"CPU used")

device used: PC GPU - NVIDIA GeForce RTX 5070 Ti


In [2]:
from transformers import AutoTokenizer
import pandas as pd

In [3]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def align_labels_tokens(labels, word_ids):
    new_labels = []
    current_word = None

    for word_id in word_ids:
        if word_id is None:
            new_labels.append(-100)

        elif word_id != current_word:
            current_word = word_id
            new_labels.append(labels[word_id])

        else:
            new_labels.append(-100)

    return new_labels

In [4]:
sample_tokens = ['Experience','with', 'micro-services','like','Kubernetes']
sample_tags = ['O','O','B-KNOWLEDGE','O','B-KNOWLEDGE']

tokenize_input = tokenizer(sample_tokens, is_split_into_words=True)
word_ids = tokenize_input.word_ids()

print(tokenize_input)
print(word_ids)

aligned_labels = align_labels_tokens(sample_tags, word_ids)

subwords = tokenizer.convert_ids_to_tokens(tokenize_input["input_ids"])

print(f"{'SUBWORD':<15} | {'ORIGINAL WORD ID':<18} | {'ALIGNED LABEL'}")
print("------------------------------------------------------------------")
for subword, word_id, label in zip(subwords, word_ids, aligned_labels):
    print(f"{'subword':<15} | {str(word_id):<18} | {label}")

{'input_ids': [101, 15843, 1114, 17599, 118, 1826, 1176, 23209, 3169, 6097, 1279, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
[None, 0, 1, 2, 2, 2, 3, 4, 4, 4, 4, None]
SUBWORD         | ORIGINAL WORD ID   | ALIGNED LABEL
------------------------------------------------------------------
subword         | None               | -100
subword         | 0                  | O
subword         | 1                  | O
subword         | 2                  | B-KNOWLEDGE
subword         | 2                  | -100
subword         | 2                  | -100
subword         | 3                  | O
subword         | 4                  | B-KNOWLEDGE
subword         | 4                  | -100
subword         | 4                  | -100
subword         | 4                  | -100
subword         | None               | -100


In [5]:
df = pd.read_parquet("../data/df_cleaned.parquet")
df.head()

,idx,tokens,tags_skill,tags_knowledge,source,split,num_skills,num_knowledge,has_overlap,tokens_length
0,1,"[Senior, QA, Engineer, (, m/f/d, ), <ORGANIZAT...","[O, O, O, O, O, O, O]","[O, O, O, O, O, O, O]",tech,train,0,0,False,7
1,1,"[<ADDRESS>, <ADDRESS>, <ADDRESS>, <ADDRESS>, <...","[O, O, O, O, O]","[O, O, O, O, O]",tech,train,0,0,False,5
2,1,"[Date, posted:, 2021-07-14]","[O, O, O]","[O, O, O]",tech,train,0,0,False,3
3,1,"[Likes:, 0, Dislikes:, 0, Love:, 0]","[O, O, O, O, O, O]","[O, O, O, O, O, O]",tech,train,0,0,False,6
4,1,"[Job, description:]","[O, O]","[O, O]",tech,train,0,0,False,2


In [6]:
unique_skills = sorted(set(tag for row in df['tags_skill'] for tag in row))
unique_knowledge = sorted(set(tag for row in df['tags_knowledge'] for tag in row))

skills2id = {tag: i for i, tag in enumerate(unique_skills)}
knowledge2id = {tag: i for i, tag in enumerate(unique_knowledge)}

id2skills = {i: tag for tag,i in skills2id.items()}
id2knowledge = {i: tag for tag,i in knowledge2id.items()}

print(skills2id)
print(knowledge2id)


{'B': 0, 'I': 1, 'O': 2}
{'B': 0, 'I': 1, 'O': 2}


In [7]:
def tokenize_align(sample, label_col, label2id):
    tokenize_inputs = tokenizer(
        sample["tokens"],
        truncation = True,
        max_length = 512,
        is_split_into_words = True,
        return_overflowing_tokens = True,
        stride = 50
    )

    labels = []

    for i, seq_idx in enumerate(tokenize_inputs.pop("overflow_to_sample_mapping")):     # map overflowing chunks back
        word_ids = tokenize_inputs.word_ids(batch_index = i)
        org_labels = sample[label_col][seq_idx]

        label_ids = [label2id[label] for label in org_labels]       # convert string labels to IDs

        aligned = align_labels_tokens(label_ids,word_ids)
        labels.append(aligned)

    tokenize_inputs["labels"] = labels
    return tokenize_inputs

In [8]:
from datasets import Dataset
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification

In [9]:
X_train = Dataset.from_pandas(df[df['split'] == "train"])
X_val = Dataset.from_pandas(df[df['split'] == "validation"])
X_test = Dataset.from_pandas(df[df['split'] == "test"])

columns_to_remove = X_train.column_names

train_tokenized_skill = X_train.map(
    lambda x: tokenize_align(x, "tags_skill", skills2id),
    batched = True,
    remove_columns= columns_to_remove
)

val_tokenized_skill = X_val.map(
    lambda x: tokenize_align(x, "tags_skill", skills2id),
    batched = True,
    remove_columns= columns_to_remove
)

test_tokenized_skill = X_test.map(
    lambda x: tokenize_align(x, "tags_skill", skills2id),
    batched = True,
    remove_columns= columns_to_remove
)

Map:   0%|          | 0/4800 [00:00<?, ? examples/s]

Map:   0%|          | 0/3174 [00:00<?, ? examples/s]

Map:   0%|          | 0/3569 [00:00<?, ? examples/s]

In [10]:
skill_model = AutoModelForTokenClassification.from_pretrained(
    'bert-base-cased',
    num_labels = len(id2skills),
    id2label = id2skills,
    label2id = skills2id
).to(device)

data_collator = DataCollatorForTokenClassification(tokenizer)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

In [11]:
import numpy as np
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report
from seqeval.scheme import IOB2

In [12]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis = 2)

    true_predictions = [
        [id2skills[p] for (p,l) in zip(prediction, label) if l != -100]
         for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [id2skills[l] for (p,l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    return {
        "Precision": precision_score(true_labels, true_predictions, mode = "strict", scheme= IOB2),
        "Recall": recall_score(true_labels, true_predictions, mode="strict", scheme=IOB2),
        "F1_Score": f1_score(true_labels,true_predictions, mode="strict", scheme=IOB2),
    }

In [13]:
from transformers import TrainingArguments, Trainer

In [14]:
training_args = TrainingArguments(
    output_dir= "../models/bert_skill",
    eval_strategy = "epoch",
    learning_rate= 2e-05,
    per_device_eval_batch_size= 16,
    per_device_train_batch_size=16,
    num_train_epochs= 3,
    weight_decay= 0.01,
    logging_steps=10
)

trainer_skill = Trainer(
    model = skill_model,
    args = training_args,
    train_dataset=train_tokenized_skill,
    eval_dataset= val_tokenized_skill,
    data_collator= data_collator,
    processing_class = tokenizer,
    compute_metrics= compute_metrics
)

print("Start Fine-Tuning Bert Skill Model")
trainer_skill.train()

Start Fine-Tuning Bert Skill Model


Epoch,Training Loss,Validation Loss,Precision,Recall,F1 Score
1,0.194107,0.169331,0.410210,0.420561,0.415321
2,0.110280,0.179668,0.471899,0.455140,0.463368
3,0.061624,0.203682,0.487081,0.475701,0.481324


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=903, training_loss=0.12858057419789062, metrics={'train_runtime': 66.7287, 'train_samples_per_second': 216.204, 'train_steps_per_second': 13.532, 'total_flos': 946012536372510.0, 'train_loss': 0.12858057419789062, 'epoch': 3.0})

Compared to baseline crf model, this is an improvement (over 2x the scores). Compared with CRF model for skill (0.3328,0.176,0.2302), the pricision climbs up to reach 0.4463, recall to 0.4152, and F1-score reach 0.4302. However, the model also starts to get overfit with the increasing epoch. at epoch = 1/3, the training loss is 0.1947 with validation loss is 0.1693. At the end of training, training loss is near perfect (0.0616), but the validation loss worse off (0.2037)

In [ ]:
bert_skill_pred = trainer_skill.predict(test_tokenized_skill)

result = compute_metrics((bert_skill_pred.predictions, bert_skill_pred.label_ids))

print("BERT Skill Extraction Model (Strict IOB2)")
print(f"Precision: {result['Precision']:.4f}")
print(f"Recall: {result['Recall']:.4f}")
print(f"F1-score: {result['F1_Score']:.4f}")

BERT Skill Extraction Model (Strict OBE2)
Precision: 0.4463
Recall: 0.4152
F1-score: 0.4302


In [20]:
result_df = pd.read_csv("../results/crf_baseline_results.csv")

new_row = pd.DataFrame([{
    'Model': 'Bert (Finetuned Standard)',
    "Entity": 'Skill',
    "Precision": round(result['Precision'], 4),
    "Recall": round(result['Recall'], 4),
    "F1_Score": round(result['F1_Score'], 4)
}])

result_df = pd.concat([result_df, new_row], ignore_index=True)
result_df.to_csv("../results/crf_baseline_results.csv", index = False)

display(result_df)

,Model,Entity,Precision,Recall,F1_Score
0,CRF Baseline (Tuned),SKILL,0.3328,0.1760,0.2302
1,CRF Baseline (Tuned),KNOWLEDGE,0.5525,0.4080,0.4694
2,CRF Baseline (Tuned),Skill,0.4463,0.4152,0.4302
3,Bert (Finetuned Standard),Skill,0.4463,0.4152,0.4302


In [25]:
def get_compute_metrics(id2label_dict):
    def compute_metrics(p):
        predictions, labels = p
        predictions = np.argmax(predictions, axis=2)

        true_predictions = [
            [id2label_dict[p] for (p, l) in zip(prediction, label) if l != -100]
            for prediction, label in zip(predictions, labels)
        ]
        true_labels = [
            [id2label_dict[l] for (p, l) in zip(prediction, label) if l != -100]
            for prediction, label in zip(predictions, labels)
        ]

        return {
            "Precision": precision_score(true_labels, true_predictions, mode="strict", scheme=IOB2),
            "Recall": recall_score(true_labels, true_predictions, mode="strict", scheme=IOB2),
            "F1_Score": f1_score(true_labels, true_predictions, mode="strict", scheme=IOB2),
        }
    return compute_metrics

---

In [21]:
columns_to_remove = X_train.column_names

train_tokenized_knowledge = X_train.map(
    lambda x: tokenize_align(x, "tags_knowledge", knowledge2id),
    batched = True,
    remove_columns= columns_to_remove
)

val_tokenized_knowledge = X_val.map(
    lambda x: tokenize_align(x, "tags_knowledge", knowledge2id),
    batched = True,
    remove_columns= columns_to_remove
)

test_tokenized_knowledge = X_test.map(
    lambda x: tokenize_align(x, "tags_knowledge", knowledge2id),
    batched = True,
    remove_columns= columns_to_remove
)

Map:   0%|          | 0/4800 [00:00<?, ? examples/s]

Map:   0%|          | 0/3174 [00:00<?, ? examples/s]

Map:   0%|          | 0/3569 [00:00<?, ? examples/s]

In [26]:
knowledge_model = AutoModelForTokenClassification.from_pretrained(
    'bert-base-cased',
    num_labels = len(id2knowledge),
    id2label = id2knowledge,
    label2id = knowledge2id
).to(device)

knowledge_training_args = TrainingArguments(
    output_dir= "../models/bert_knowledge",
    eval_strategy = "epoch",
    learning_rate= 2e-05,
    per_device_eval_batch_size= 16,
    per_device_train_batch_size=16,
    num_train_epochs= 3,
    weight_decay= 0.01,
    logging_steps=10
)

trainer_knowledge = Trainer(
    model = knowledge_model,
    args = knowledge_training_args,
    train_dataset=train_tokenized_knowledge,
    eval_dataset= val_tokenized_knowledge,
    data_collator= data_collator,
    processing_class = tokenizer,
    compute_metrics= get_compute_metrics(id2knowledge)
)

print("Start Fine-Tuning Bert Knowledge Model")
trainer_knowledge.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

Start Fine-Tuning Bert Knowledge Model


Epoch,Training Loss,Validation Loss,Precision,Recall,F1 Score
1,0.087082,0.091895,0.519276,0.603843,0.558376
2,0.063754,0.087992,0.522936,0.625801,0.569763
3,0.027917,0.095839,0.526070,0.618481,0.568545


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=903, training_loss=0.07817183735296708, metrics={'train_runtime': 68.9953, 'train_samples_per_second': 209.101, 'train_steps_per_second': 13.088, 'total_flos': 946012536372510.0, 'train_loss': 0.07817183735296708, 'epoch': 3.0})

In [27]:
bert_knowledge_pred = trainer_knowledge.predict(test_tokenized_knowledge)

result = compute_metrics((bert_knowledge_pred.predictions, bert_knowledge_pred.label_ids))

print("BERT Knowledge Extraction Model (Strict IOB2)")
print(f"Precision: {result['Precision']:.4f}")
print(f"Recall: {result['Recall']:.4f}")
print(f"F1-score: {result['F1_Score']:.4f}")

BERT Knowledge Extraction Model (Strict IOB2)
Precision: 0.5716
Recall: 0.6559
F1-score: 0.6109


Compared to the baseline CRF model for knowledge (0.5525 Precision, 0.4080 Recall, 0.4694 F1-Score), BERT's precision climbs to 0.5716, recall to 0.6559, and the F1-score reaches 0.6109. However, the model does suffer from mild overfitting with increasing epochs. At Epoch 1, the training loss (0.087082) and validation loss (0.091895) are nearly identical. But by the end of training (Epoch 3), the training loss plummets to 0.027917 while the validation loss reaches up to 0.095839. This divergence indicates the model is starting to memorize the training data rather than generalizing. Finally, BERT's higher performance on knowledge compared to skills makes sense: knowledge entities are shorter and more concise (median 1 word), while multi-word skill spans (median 4 words) remain much harder for a general language model to delineate boundary-by-boundary.

In [28]:
result_df = pd.read_csv("../results/crf_baseline_results.csv")

new_row = pd.DataFrame([{
    'Model': 'Bert (Finetuned Standard)',
    "Entity": 'Knowledge',
    "Precision": round(result['Precision'], 4),
    "Recall": round(result['Recall'], 4),
    "F1_Score": round(result['F1_Score'], 4)
}])

result_df = pd.concat([result_df, new_row], ignore_index=True)
result_df.to_csv("../results/crf_baseline_results.csv", index = False)

display(result_df)

,Model,Entity,Precision,Recall,F1_Score
0,CRF Baseline (Tuned),SKILL,0.3328,0.1760,0.2302
1,CRF Baseline (Tuned),KNOWLEDGE,0.5525,0.4080,0.4694
2,Bert (Finetuned Standard),Skill,0.4463,0.4152,0.4302
3,Bert (Finetuned Standard),Knowledge,0.5807,0.6678,0.6212
4,Bert (Finetuned Standard),Knowledge,0.5716,0.6559,0.6109
